# Práctica N.° 07 — Tablas Hash

Este notebook contiene las actividades de Python: **Actividad 1, 2, 3, 4 y 6** (más un pequeño complemento que compara las tablas propias con `dict`).
La **Actividad 5** (C++) está en el archivo `.cpp`.


In [ ]:
import random 
import string
import math
import time
import re
from collections import defaultdict

random.seed(42)   # resultados reproducibles; quita esta línea si quieres datos distintos en cada corrida

## Actividad 1 — Funciones hash y análisis de distribución

Se comparan la **hash por módulo** (claves enteras) y la **hash polinomial** (strings).
Para medir qué tan bien se reparten las claves usamos el **coeficiente de variación (CV)** de cuántas claves caen en cada celda.

> Referencia: aunque la función hash sea buena, el reparto nunca es perfecto. Para claves aleatorias, el CV esperado es aproximadamente `sqrt(m / n)`. Un CV mucho mayor que ese valor indica mala distribución.

In [2]:
def hash_modulo(k: int, m: int) -> int:
    return k % m


def hash_polinomial(s: str, m: int, p: int = 31) -> int:
    """Hash polinomial para strings: O(|s|)."""
    h = 0
    potencia = 1
    for c in s:
        h = (h + ord(c) * potencia) % m
        potencia = (potencia * p) % m
    return h


def medir_distribucion(claves, m, hash_fn) -> dict:
    """Cuenta cuántos elementos caen en cada celda y calcula estadísticas."""
    tabla = [0] * m
    for k in claves:
        tabla[hash_fn(k, m)] += 1
    ocupadas = sum(1 for c in tabla if c > 0)
    promedio = len(claves) / m
    varianza = sum((c - promedio) ** 2 for c in tabla) / m
    cv = math.sqrt(varianza) / promedio if promedio > 0 else 0
    return {'ocupadas': ocupadas, 'max_colisiones': max(tabla), 'cv': cv}

In [3]:
m = 101   # primo
n = 500

# Hash de enteros
claves_int = [random.randint(0, 1_000_000) for _ in range(n)]
stats = medir_distribucion(claves_int, m, hash_modulo)
print(f"Hash enteros (m={m}, n={n}): celdas_ocupadas={stats['ocupadas']}, "
      f"max_col={stats['max_colisiones']}, CV={stats['cv']:.3f}")

# Hash de strings
claves_str = [''.join(random.choices(string.ascii_lowercase, k=8)) for _ in range(n)]
stats_s = medir_distribucion(claves_str, m, hash_polinomial)
print(f"Hash strings (m={m}, n={n}): celdas_ocupadas={stats_s['ocupadas']}, "
      f"max_col={stats_s['max_colisiones']}, CV={stats_s['cv']:.3f}")

print(f"\nCV esperado para un reparto aleatorio ideal: {math.sqrt(m / n):.3f}")
print("(un CV cercano a ese valor indica buena distribución)")

Hash enteros (m=101, n=500): celdas_ocupadas=100, max_col=10, CV=0.432
Hash strings (m=101, n=500): celdas_ocupadas=101, max_col=13, CV=0.483

CV esperado para un reparto aleatorio ideal: 0.449
(un CV cercano a ese valor indica buena distribución)


## Actividad 2 — `TablaHashChaining` (encadenamiento)

Cada celda de la tabla guarda una **lista enlazada**. Si ya tienes tu `ListaEnlazada` de la Práctica 5, puedes usarla en lugar de la que está aquí (solo necesita buscar, agregar y eliminar por clave).

Operaciones: `insertar`, `buscar`, `eliminar` y **rehash automático** cuando α > 0.75.

In [4]:
def es_primo(x: int) -> bool:
    if x < 2:
        return False
    for i in range(2, int(x ** 0.5) + 1):
        if x % i == 0:
            return False
    return True


def siguiente_primo(n: int) -> int:
    while not es_primo(n):
        n += 1
    return n


class Nodo:
    """Nodo de la lista enlazada."""
    def __init__(self, clave, valor):
        self.clave = clave
        self.valor = valor
        self.siguiente = None


class ListaEnlazada:
    """Lista enlazada simple de pares (clave, valor)."""
    def __init__(self):
        self.cabeza = None

    def buscar_nodo(self, clave):
        actual = self.cabeza
        while actual is not None:
            if actual.clave == clave:
                return actual
            actual = actual.siguiente
        return None

    def agregar(self, clave, valor):
        """Inserta al inicio: O(1)."""
        nodo = Nodo(clave, valor)
        nodo.siguiente = self.cabeza
        self.cabeza = nodo

    def eliminar(self, clave) -> bool:
        anterior = None
        actual = self.cabeza
        while actual is not None:
            if actual.clave == clave:
                if anterior is None:
                    self.cabeza = actual.siguiente
                else:
                    anterior.siguiente = actual.siguiente
                return True
            anterior = actual
            actual = actual.siguiente
        return False

    def nodos(self):
        """Recorre los nodos de la lista (se usa en el rehash)."""
        actual = self.cabeza
        while actual is not None:
            yield actual
            actual = actual.siguiente

In [5]:
class TablaHashChaining:
    """Tabla hash con encadenamiento. Rehash automático cuando alpha > 0.75."""
    UMBRAL = 0.75

    def __init__(self, m: int = 11):
        self._m = m
        self._n = 0
        self._tabla = [ListaEnlazada() for _ in range(m)]
        self._rehashes = 0

    def _hash(self, clave) -> int:
        return hash(clave) % self._m

    def insertar(self, clave, valor) -> None:
        lista = self._tabla[self._hash(clave)]
        nodo = lista.buscar_nodo(clave)
        if nodo is not None:          # la clave ya existe: se actualiza el valor
            nodo.valor = valor
            return
        lista.agregar(clave, valor)
        self._n += 1
        if self._n / self._m > self.UMBRAL:
            self._rehash()

    def buscar(self, clave):
        nodo = self._tabla[self._hash(clave)].buscar_nodo(clave)
        return nodo.valor if nodo is not None else None

    def eliminar(self, clave) -> bool:
        if self._tabla[self._hash(clave)].eliminar(clave):
            self._n -= 1
            return True
        return False

    def _rehash(self):
        vieja = self._tabla
        self._m = siguiente_primo(self._m * 2)
        self._tabla = [ListaEnlazada() for _ in range(self._m)]
        self._rehashes += 1
        for lista in vieja:
            for nodo in lista.nodos():
                self._tabla[self._hash(nodo.clave)].agregar(nodo.clave, nodo.valor)
        # _n no cambia: son los mismos elementos

    def __len__(self):
        return self._n

    @property
    def factor_carga(self):
        return self._n / self._m

In [6]:
# Prueba
ciudades = [('Puno', 3825), ('Juliaca', 1100), ('Arequipa', 2335), ('Lima', 154),
            ('Cusco', 3400), ('Iquitos', 106), ('Trujillo', 34), ('Huancayo', 3271)]

t = TablaHashChaining(m=7)
for ciudad, alt in ciudades:
    t.insertar(ciudad, alt)
print(f'alpha={t.factor_carga:.2f}, rehashes={t._rehashes}')

assert t.buscar('Puno') == 3825
assert t.eliminar('Lima')
assert t.buscar('Lima') is None
assert t.buscar('Cusco') == 3400
t.insertar('Puno', 3830)                 # actualizar una clave existente
assert t.buscar('Puno') == 3830 and len(t) == 7
print('TablaHashChaining: OK ✓')

alpha=0.47, rehashes=1
TablaHashChaining: OK ✓


## Actividad 3 — `TablaHashOpenAddressing` (sondeo lineal con marcadores)

Todos los elementos viven dentro del arreglo. Si la celda está ocupada se prueba la siguiente: `(hash(k) + j) mod m`.

Al eliminar se deja un marcador `DELETED` (no `VACIA`) para no cortar las cadenas de sondeo.

Detalles que cuida esta versión:
- `insertar` recorre la cadena **hasta una celda `VACIA`** antes de insertar, para no duplicar una clave que ya existe más adelante.
- Los `DELETED` también cuentan para el factor de carga (se llevan en `_borrados`); si no, se acumularían y las búsquedas se volverían lentas.

In [7]:
class TablaHashOpenAddressing:
    """Sondeo lineal con marcadores DELETED para eliminación correcta."""
    VACIA = object()     # celda nunca usada
    DELETED = object()   # celda eliminada (no corta la cadena de sondeo)
    UMBRAL = 0.70

    def __init__(self, m: int = 11):
        self._m = m
        self._n = 0            # elementos vivos
        self._borrados = 0     # marcadores DELETED
        self._tabla = [self.VACIA] * m
        self._rehashes = 0

    def _hash(self, clave, j: int) -> int:
        return (hash(clave) + j) % self._m     # sondeo lineal

    def insertar(self, clave, valor) -> None:
        if (self._n + self._borrados) / self._m > self.UMBRAL:
            self._rehash()

        primera_libre = None       # primera celda VACIA o DELETED que encontremos
        for j in range(self._m):
            idx = self._hash(clave, j)
            celda = self._tabla[idx]
            if celda is self.VACIA:
                if primera_libre is None:
                    primera_libre = idx
                break                              # aquí termina la cadena: la clave no existe
            if celda is self.DELETED:
                if primera_libre is None:
                    primera_libre = idx
            elif celda[0] == clave:                # la clave ya existe: se actualiza
                self._tabla[idx] = (clave, valor)
                return

        if primera_libre is None:
            raise RuntimeError('Tabla llena')
        if self._tabla[primera_libre] is self.DELETED:
            self._borrados -= 1
        self._tabla[primera_libre] = (clave, valor)
        self._n += 1

    def buscar(self, clave):
        for j in range(self._m):
            idx = self._hash(clave, j)
            celda = self._tabla[idx]
            if celda is self.VACIA:
                return None                        # cadena interrumpida → no existe
            if celda is not self.DELETED and celda[0] == clave:
                return celda[1]
        return None

    def eliminar(self, clave) -> bool:
        for j in range(self._m):
            idx = self._hash(clave, j)
            celda = self._tabla[idx]
            if celda is self.VACIA:
                return False
            if celda is not self.DELETED and celda[0] == clave:
                self._tabla[idx] = self.DELETED    # se marca, NO se deja VACIA
                self._n -= 1
                self._borrados += 1
                return True
        return False

    def _rehash(self):
        vieja = self._tabla
        self._m = siguiente_primo(max(11, 2 * self._n))   # deja la tabla cerca de alpha = 0.5
        self._tabla = [self.VACIA] * self._m
        self._n = 0
        self._borrados = 0
        self._rehashes += 1
        for celda in vieja:
            if celda is not self.VACIA and celda is not self.DELETED:
                self.insertar(celda[0], celda[1])

    def __len__(self):
        return self._n

    @property
    def factor_carga(self):
        return self._n / self._m

In [8]:
# Prueba 1: las ciudades de la Actividad 2
t2 = TablaHashOpenAddressing(m=13)
for ciudad, alt in ciudades:
    t2.insertar(ciudad, alt)
assert t2.buscar('Puno') == 3825
assert t2.eliminar('Juliaca')
assert t2.buscar('Juliaca') is None
assert t2.buscar('Puno') == 3825      # la cadena no se rompió
assert t2.buscar('Lima') == 154

# Prueba 2: claves que colisionan a propósito (con m=10, todas caen en la celda 0)
t3 = TablaHashOpenAddressing(m=10)
for k in [10, 20, 30, 40, 50]:
    t3.insertar(k, 'v' + str(k))
t3.eliminar(20)                        # queda un DELETED en medio de la cadena
assert t3.buscar(30) == 'v30'          # sigue encontrándose gracias al marcador
assert t3.buscar(50) == 'v50'
t3.insertar(40, 'nuevo')               # 40 ya existe: debe actualizarse, no duplicarse
assert t3.buscar(40) == 'nuevo' and len(t3) == 4

# Prueba 3: rehash con muchos elementos y eliminaciones
t4 = TablaHashOpenAddressing(m=5)
for i in range(100):
    t4.insertar(i, i * i)
for i in range(0, 100, 2):
    t4.eliminar(i)
assert all(t4.buscar(i) is None for i in range(0, 100, 2))
assert all(t4.buscar(i) == i * i for i in range(1, 100, 2))
print(f'rehashes={t4._rehashes}, elementos={len(t4)}, alpha={t4.factor_carga:.2f}')
print('TablaHashOpenAddressing: OK ✓')

rehashes=8, elementos=50, alpha=0.26
TablaHashOpenAddressing: OK ✓


## Actividad 4 — Miniestudio experimental: factor de carga vs colisiones

Para cada configuración `(n, m)` se hacen **10 repeticiones** insertando `n` strings aleatorios y se promedian los resultados.

Qué se cuenta:
- **Encadenamiento:** una *colisión* ocurre cuando se inserta en una celda que ya tenía elementos.
- **Direccionamiento abierto (sondeo lineal):** se cuenta **cada sondeo fallido** (cada vez que hay que pasar a la celda siguiente). Por eso este número suele ser mayor: mide el costo total de las colisiones, no solo cuántas hubo.

Con α > 1 el direccionamiento abierto **no puede** funcionar (hay más elementos que celdas), así que en esas filas aparece `N/A`.

Se usa la hash polinomial de la Actividad 1 para que los resultados sean reproducibles.
La columna *teórico* es el valor esperado para encadenamiento con una hash ideal: `n − m·(1 − e^(−α))`.

In [9]:
def medir_colisiones_reales(n_elementos, m_tabla, n_rep=10):
    """Promedia, en n_rep repeticiones, las colisiones de ambos métodos."""
    total_ch = 0
    total_oa = 0
    oa_posible = n_elementos <= m_tabla     # con n > m no cabe en direccionamiento abierto

    for rep in range(n_rep):
        celdas_ch = [0] * m_tabla           # encadenamiento: basta contar cuántos hay en cada celda
        tabla_oa = [None] * m_tabla         # direccionamiento abierto

        for i in range(n_elementos):
            k = ''.join(random.choices(string.ascii_lowercase, k=8))
            idx = hash_polinomial(k, m_tabla)

            # Encadenamiento
            if celdas_ch[idx] > 0:
                total_ch += 1
            celdas_ch[idx] += 1

            # Direccionamiento abierto (sondeo lineal)
            if oa_posible:
                j = 0
                while tabla_oa[(idx + j) % m_tabla] is not None:
                    j += 1
                    total_oa += 1
                tabla_oa[(idx + j) % m_tabla] = k

    col_ch = total_ch / n_rep
    col_oa = total_oa / n_rep if oa_posible else None
    return col_ch, col_oa

In [10]:
configs = [(250, 1000), (500, 1000), (750, 1000), (1000, 1000), (1500, 1000), (2000, 1000)]

print(f"{'alpha':>6} {'n':>6} {'m':>6} {'Col_Chain':>10} {'Teórico':>9} {'Col_OpenAddr':>13} {'%_chain':>8} {'%_open':>8}")
for n, m in configs:
    cc, co = medir_colisiones_reales(n, m, n_rep=10)
    alpha = n / m
    teorico = n - m * (1 - math.exp(-alpha))
    if co is None:
        print(f"{alpha:>6.2f} {n:>6} {m:>6} {cc:>10.1f} {teorico:>9.1f} {'N/A':>13} {cc / n * 100:>7.1f}% {'N/A':>8}")
    else:
        print(f"{alpha:>6.2f} {n:>6} {m:>6} {cc:>10.1f} {teorico:>9.1f} {co:>13.1f} {cc / n * 100:>7.1f}% {co / n * 100:>7.1f}%")

 alpha      n      m  Col_Chain   Teórico  Col_OpenAddr  %_chain   %_open
  0.25    250   1000       29.8      28.8          44.9    11.9%    18.0%
  0.50    500   1000      106.7     106.5         246.2    21.3%    49.2%
  0.75    750   1000      220.1     222.4        1066.7    29.3%   142.2%
  1.00   1000   1000      373.0     367.9       17996.8    37.3%  1799.7%
  1.50   1500   1000      723.1     723.1           N/A    48.2%      N/A
  2.00   2000   1000     1139.9    1135.3           N/A    57.0%      N/A


> **Para completar después:** la tabla de resultados del informe y la pregunta *«¿a qué valor de α comienza a degradarse significativamente el sondeo lineal comparado con el encadenamiento?»* (se responden con los números de la celda anterior).

## Complemento — Tablas propias vs `dict` de Python

El objetivo específico 5 pide comparar el rendimiento de las implementaciones propias contra `dict`. Se insertan y buscan las mismas claves en las tres estructuras.

In [11]:
def benchmark(nombre, crear, insertar, buscar, claves):
    tabla = crear()

    t0 = time.perf_counter()
    for i, k in enumerate(claves):
        insertar(tabla, k, i)
    t_insertar = time.perf_counter() - t0

    t0 = time.perf_counter()
    for k in claves:
        buscar(tabla, k)
    t_buscar = time.perf_counter() - t0

    print(f'{nombre:<24} insertar: {t_insertar * 1000:8.1f} ms    buscar: {t_buscar * 1000:8.1f} ms')


def ins_dict(d, k, v):
    d[k] = v

def bus_dict(d, k):
    return d.get(k)


N = 20000
claves_bench = [''.join(random.choices(string.ascii_lowercase, k=8)) for _ in range(N)]
print(f'{N} claves de 8 letras\n')

benchmark('TablaHashChaining', TablaHashChaining,
          lambda t, k, v: t.insertar(k, v), lambda t, k: t.buscar(k), claves_bench)
benchmark('TablaHashOpenAddressing', TablaHashOpenAddressing,
          lambda t, k, v: t.insertar(k, v), lambda t, k: t.buscar(k), claves_bench)
benchmark('dict de Python', dict, ins_dict, bus_dict, claves_bench)

20000 claves de 8 letras

TablaHashChaining        insertar:     61.5 ms    buscar:      4.3 ms
TablaHashOpenAddressing  insertar:     45.1 ms    buscar:      7.8 ms
dict de Python           insertar:      1.4 ms    buscar:      0.9 ms


## Actividad 6 — Índice invertido con tabla hash

Un **índice invertido** asocia cada palabra con el conjunto de documentos donde aparece (así funcionan los buscadores). Aquí el diccionario de Python (`dict` y `set`) hace de tabla hash.

Dos detalles cuidados:
- Las palabras se limpian con una expresión regular (sin signos como `:`), y la consulta se procesa igual que los títulos.
- En `buscar` se trabaja sobre una **copia** del primer conjunto, porque `&=` modifica el conjunto en el lugar y dañaría el índice.

In [12]:
# Catálogo de la Biblioteca Central UNA-PUNO
DOCUMENTOS = {
    1: 'Algoritmos y estructuras de datos en Python para ingeniería',
    2: 'Análisis de complejidad computacional y algoritmos recursivos',
    3: 'Estructuras de datos lineales: listas pilas y colas',
    4: 'Tablas hash y funciones de dispersión para bases de datos',
    5: 'Árboles binarios de búsqueda y algoritmos de recorrido',
    6: 'Grafos y algoritmos de caminos mínimos en redes viales',
    7: 'Fundamentos de programación en C++ para sistemas embebidos',
}

STOP_WORDS = {'y', 'de', 'en', 'para', 'el', 'la', 'los', 'las', 'a', 'e', 'o', 'un', 'una'}


def tokenizar(texto: str) -> list:
    """Pasa a minúsculas y separa en palabras (sin signos de puntuación)."""
    return re.findall(r'\w+', texto.lower())


def construir_indice_invertido(docs: dict) -> dict:
    """
    Construye un índice invertido: {palabra: {doc_id, ...}}
    Complejidad: O(N), donde N = número total de palabras en todos los documentos.
    """
    indice = defaultdict(set)
    for doc_id, titulo in docs.items():
        for palabra in tokenizar(titulo):
            if palabra not in STOP_WORDS and len(palabra) > 2:
                indice[palabra].add(doc_id)
    return dict(indice)


def buscar(indice: dict, consulta: str) -> list:
    """Búsqueda multi-término: AND de todos los términos."""
    palabras = tokenizar(consulta)
    if not palabras:
        return []
    resultados = set(indice.get(palabras[0], set()))    # copia: no se toca el índice
    for p in palabras[1:]:
        resultados &= indice.get(p, set())
    return sorted(resultados)

In [13]:
indice = construir_indice_invertido(DOCUMENTOS)
print(f'Vocabulario: {len(indice)} términos únicos')
print(f'Documentos promedio por término: {sum(len(v) for v in indice.values()) / len(indice):.2f}')

consultas = ['algoritmos', 'estructuras datos', 'tablas hash', 'árboles grafos']
for q in consultas:
    ids = buscar(indice, q)
    titulos = [DOCUMENTOS[i] for i in ids]
    print(f'\nConsulta: {q!r}')
    print(f'  Resultados ({len(ids)}): {titulos}')

# Comprobación: las consultas no modificaron el índice
assert indice == construir_indice_invertido(DOCUMENTOS)
print('\nÍndice intacto tras las consultas ✓')

Vocabulario: 31 términos únicos
Documentos promedio por término: 1.19

Consulta: 'algoritmos'
  Resultados (4): ['Algoritmos y estructuras de datos en Python para ingeniería', 'Análisis de complejidad computacional y algoritmos recursivos', 'Árboles binarios de búsqueda y algoritmos de recorrido', 'Grafos y algoritmos de caminos mínimos en redes viales']

Consulta: 'estructuras datos'
  Resultados (2): ['Algoritmos y estructuras de datos en Python para ingeniería', 'Estructuras de datos lineales: listas pilas y colas']

Consulta: 'tablas hash'
  Resultados (1): ['Tablas hash y funciones de dispersión para bases de datos']

Consulta: 'árboles grafos'
  Resultados (0): []

Índice intacto tras las consultas ✓
